<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Data Cleaning in PySpark

*Session 5 · Notebook 02.02 · Lecture · Student version*

## Overview

Before any analysis or model, the data has to be trustworthy. This short notebook is the PySpark twin of the Session 3 **Data Cleaning** lecture, in the same order: a first inspection, missing data, type conversion, transforming values, cleaning text, and duplicates, with one extra step that matters at scale: **rule checks** that set bad records aside instead of letting them through.

Each step is taught on a tiny table of loan applications with deliberate problems; the rule checks are then applied to the **Superstore** data. The practical that follows (02.03) uses the same toolkit on the Titanic passengers.

## Learning Objectives

By the end of this notebook you will be able to:

- Inspect a DataFrame: row counts, null counts per column and summary statistics, and tell `null` from `NaN`.
- Drop or fill missing values, for all columns or a chosen subset.
- Convert types safely and find the values that fail a conversion.
- Transform values with built-in functions and `replace` (the PySpark way to do what `.apply()` did in Session 3).
- Clean text columns: trim, change case, replace and extract patterns.
- Find and remove duplicate rows, on all columns or on a business key.
- Plan a cleaning job as a list of steps, and check the effect of each step before moving on.
- Express data rules as conditions and split records into valid and quarantined sets.

## Prerequisites

- Notebook 01.02 (Part 2, the PySpark walkthrough: select, filter, `withColumn`, `groupBy`).
- The Session 3 data cleaning lecture: every section here has a twin there.

## Index

1. [Why this matters](#why)
2. [Setup](#setup)
3. [First inspection: what is in the data?](#profile)
4. [Missing values: find, drop or fill](#missing)
5. [Data type conversion](#types)
6. [Transforming values](#transform)
7. [Cleaning text columns](#text)
8. [Duplicates](#dups)
9. [Rule checks: a quality report for Superstore](#rules)
10. [Exercises](#exercises)
11. [Key Takeaways](#takeaways)
12. [Conclusion](#conclusion)
13. [Further Reading & Resources](#reading)

<a id="why"></a>
# Section 1: Why this matters

A model or report is only as good as its inputs. At scale, problems that are obvious in a spreadsheet become invisible: nobody scrolls through 200 million rows. Data cleaning has to be **code**, run every time the data arrives.

| Problem | What goes wrong | Spark tool |
|---|---|---|
| Missing values | Averages and models silently skip or mis-handle rows | `isNull`, `na.drop`, `na.fill` |
| Wrong types | Numbers stored as text cannot be summed; dates cannot be compared | `try_cast`, `try_to_timestamp` |
| Inconsistent values | `"Card"`, `"card"` and `" CARD "` counted as three products | `trim`, `upper`, `replace` |
| Duplicates | Totals are inflated, customers counted twice | `dropDuplicates` |
| Invalid values | A negative amount or an unknown code passes into reports | Rule conditions + quarantine |

The guiding principle: **measure first, then decide**. Count the problem before you fix it, and never drop data without knowing how much you are dropping.

<a id="setup"></a>
# Section 2: Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # finds config.py next to this notebook or one level up

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from config import pandas_bridge, session_datasets, session_datasets_http

# On Databricks this returns the session that already exists; on a laptop it starts one.
spark = SparkSession.builder.appName("05_02_02_DataCleaning").getOrCreate()
print("Spark version:", spark.version)

A small table of loan applications with deliberate problems: missing values, an exact duplicate, a repeated application id, an amount stored as text that is not a number, a negative amount, an unknown product, a badly formatted date, and product names typed inconsistently (spaces, case).

In [ ]:
loans = spark.createDataFrame(
    [
        ("A001", "2024-01-05", "Mortgage",   "250000", 41,   "Approved", "  alice SMITH"),
        ("A002", "2024-01-07", "Personal",   "12000",  None, "Approved", "Bob Jones"),
        ("A003", "2024-01-09", "card",       "abc",    35,   "Declined", "chen  wei "),
        ("A004", "2024-01-10", "Personal",   "-500",   29,   None,       "Dana O'Neil"),
        ("A005", "10/01/2024", "Auto",       "18000",  52,   "Approved", "EVE ADAMS"),
        ("A006", "2024-01-12", "Crypto",     "9000",   23,   "Approved", "Frank Li"),
        ("A007", None,         " Mortgage ", "310000", 47,   "Approved", "Grace Hopper"),
        ("A002", "2024-01-07", "Personal",   "12000",  None, "Approved", "Bob Jones"),     # exact duplicate of A002
        ("A003", "2024-01-15", "Card",       "4000",   35,   "Approved", "Chen Wei"),      # same id, different details
    ],
    "app_id STRING, app_date STRING, product STRING, amount STRING, age INT, decision STRING, applicant STRING",
)
loans.show(truncate=False)

**The plan.** Cleaning is a sequence of small, deliberate steps, each one measured before and after. These are the problems in the table, and the section that deals with each:

| Problem in `loans` | Example | Step | Section |
|---|---|---|---|
| Missing values | `age` of A002, `decision` of A004, `app_date` of A007 | Count them, then drop or fill | 3 and 4 |
| Wrong types | `amount` is text; `app_date` is text | Convert, and list the values that fail | 5 |
| Values to recode | decision `Approved` / `Declined` as a 0/1 flag | Map old values to new ones | 6 |
| Messy text | `" Mortgage "`, `card` / `Card`, `"  alice SMITH"` | Trim, squeeze spaces, fix the case | 7 |
| Duplicates | A002 twice (identical); A003 twice (different details) | Remove exact copies, then look at repeated ids | 8 |
| Values that break a rule | amount `-500`, product `Crypto` | Rules and quarantine (on Superstore) | 9 |

<a id="profile"></a>
# Section 3: First inspection: what is in the data?

**Definition.** A **first inspection** (or **profile**) summarises a dataset's shape and content before using it: how many rows, which types, how many missing values per column, and the range of each numeric column.

**Example.** For the loans table: 9 rows, `amount` stored as text, nulls in `app_date`, `age` and `decision`.

**Analogy.** A health check before surgery: measure first, act second.

**Explanation.** The null count per column uses a pattern worth memorising: for every column, count the rows where it is null. `F.count` ignores nulls, so `F.count(F.when(cond, 1))` counts only the rows where `cond` is true. It is the twin of pandas' `df.isnull().sum()`.

In [ ]:
# How big is the table?
# Which type does each column have? (amount and app_date are text: Section 5 converts them)
# How many nulls in each column? For every column c, the list comprehension builds one count:
#   F.when(F.col(c).isNull(), 1) gives 1 when c is null, and null otherwise
#   F.count(...) counts only non-null values, so it counts the rows where c is null
#   .alias(c) names the result after the column

`describe()` gives count, mean, standard deviation, min and max; `summary()` adds percentiles. The `count` row is itself a missing-value check: it counts non-null values.

In [ ]:
# Summary statistics of the age column; count = 8 of 9 rows, so one age is missing

<a id="missing"></a>
# Section 4: Missing values: find, drop or fill

**Definition.** Once you know where values are missing, you either **drop** the affected rows or **fill** (impute) them with a sensible value.

**Example.** Drop applications with no date (we cannot place them in time); fill a missing decision with `"Unknown"`; fill a missing age with the median age.

**Analogy.** A survey with blank answers: discard incomplete forms, or write "not answered" and carry on. Which is right depends on how many forms are blank and how much the answer matters.

**Explanation.** The `.na` interface does both, like pandas' `dropna` and `fillna`:

```python
df.na.drop()                                  # drop rows with ANY null          (dropna())
df.na.drop(subset=["app_date"])               # only look at chosen columns      (dropna(subset=...))
df.na.fill("Unknown", subset=["decision"])    # fill text columns                (fillna(...))
df.na.fill({"decision": "Unknown", "age": 0}) # a different value per column
```

For numbers, a statistic is usually better than zero. Compute it with an aggregation, bring the single number back with `.first()[0]`, then fill. The **median** (`percentile_approx`, cheap on big data) is safer than the mean when the data is skewed.

**`null` versus `NaN`.** Spark has two kinds of "missing" for numbers: `null` means *no value*; `NaN` ("not a number") is a special floating-point **value** (from `0.0 / 0.0`, or data arriving from pandas). `isNull()` does **not** catch `NaN`, so for decimal columns check both with `F.isnan`. (The course `pandas_bridge` helper converts `NaN` to `null` for you.)

**What we want to do.**

1. **Measure** how many rows each way of dropping would keep, before deciding anything.
2. Drop only the rows we cannot use: an application without a date cannot be placed in time.
3. Fill the rest: a missing `decision` becomes `"Unknown"`, a missing `age` gets the median age.

In [ ]:
# Step 1: measure before deciding. How many rows would each option keep?

In [ ]:
# Step 2: drop only the rows without an application date
# Step 3a: the value used to fill missing ages, the median.
# percentile_approx(column, 0.5) is the median; .first()[0] takes the single number out of the
# one-row result and brings it back to Python
# Step 3b: fill each column with its own value (a dictionary of column -> value)

In [ ]:
# null versus NaN: a tiny table with one real value, one null and one NaN
demo = spark.createDataFrame([(1.0,), (None,), (float("nan"),)], "x DOUBLE")

In [ ]:
# isNull() finds the null only; F.isnan() finds the NaN only: check both on decimal columns

#### Try it yourself

Drop the rows of `loans` where `app_date` is missing, then fill a missing `decision` with `"Pending"`. Save the result as `loans_step1` and confirm with a null count that `app_date` and `decision` no longer contain nulls.

In [ ]:
# Your turn

<a id="types"></a>
# Section 5: Data type conversion

**Definition.** A **type problem** is a value stored in the wrong type (numbers or dates as text). An **invalid value** is one that cannot be converted at all, such as `"abc"` in an amount column or `"10/01/2024"` where `yyyy-MM-dd` is expected.

**Example.** Convert `amount` to a number and `app_date` to a date, then find the rows where the conversion failed.

**Analogy.** Customs at a border: each value must show the right papers to enter the new type. Depending on the rules in force, a value without papers either stops the whole queue (an error) or is waved through with a blank passport (`null`).

**Explanation.** Which rule applies depends on Spark's **ANSI mode** (notebook 01.02). With ANSI mode on (the default on Databricks serverless and in Spark 4), a plain `cast` or `to_date` on a bad value makes the whole query **fail**. For data you have not checked yet, use the **`try_` functions**, which return `null` for a bad value instead of failing (the twin of pandas' `pd.to_numeric(..., errors="coerce")`):

| Conversion | Strict (may fail under ANSI) | Tolerant (bad value becomes null) |
|---|---|---|
| Text to number | `F.col("x").cast("double")` | `F.expr("try_cast(x AS DOUBLE)")` |
| Text to date | `F.to_date("x", "yyyy-MM-dd")` | `F.expr("to_date(try_to_timestamp(x, 'yyyy-MM-dd'))")` |

The check is then: **the original was not null, but the converted value is**. Those rows are conversion failures.

**What we want to do.**

1. Convert `amount` to a number and `app_date` to a date, into **new** columns, so the original text is still there to compare.
2. List the rows where the conversion failed: the original had a value, the converted column is null.
3. Decide what to do with them (fix at the source, or quarantine as in Section 9).

In [ ]:
# Step 1: convert into new columns, keeping the originals
# text -> number; a value such as "abc" becomes null instead of stopping the query
# text -> date in the format yyyy-MM-dd; "10/01/2024" does not fit, so it becomes null
# Step 2: a failed conversion = the original had a value, but the converted column is null

<a id="transform"></a>
# Section 6: Transforming values

**Definition.** **Transforming values** means mapping existing values to new ones: recoding categories, grouping numbers into bands, replacing codes with labels.

**Example.** Turn the decision into a 0/1 flag; band the applicants by age; rename the product `"Personal"` to `"Personal loan"`.

**Analogy.** A translation table pinned next to the spreadsheet: every old value is looked up and replaced by its new value.

**Explanation.** In Session 3 you wrote these with `.apply()` and a Python function. In PySpark, as the walkthrough (notebook 01.02, Part 2) showed, you reach for **built-in functions first**:

| pandas (Session 3) | PySpark |
|---|---|
| `df["x"].apply(lambda v: 1 if v == "Approved" else 0)` | `F.when(F.col("x") == "Approved", 1).otherwise(0)` |
| `df["x"].replace({"old": "new"})` / `df["x"].map(dict)` | `df.replace({"old": "new"}, subset=["x"])` |
| `pd.cut(df["age"], bins=[...], labels=[...])` | a chain of `F.when(...).when(...).otherwise(...)` |
| a custom Python function with `.apply()` | a Python UDF, only if no built-in can do it (slow) |

**What we want to do.** Three new or changed columns: a 0/1 flag `approved` from the decision, an `age_band` from the age, and the product `"Personal"` renamed to `"Personal loan"`.

In [ ]:
# 1. a flag: 1 when the decision is "Approved", 0 for anything else (including a missing decision)
# 2. bands: the conditions are checked in order and the first one that is true wins
# 3. recode one value with a dictionary {old: new}, only in the product column

<a id="text"></a>
# Section 7: Cleaning text columns

**Definition.** Text arrives with stray spaces, inconsistent capitals and mixed formats. **Cleaning** brings each value to one standard form, so that equal things compare as equal.

**Example.** `"card"`, `"Card"` and `" Mortgage "` in `product`; `"  alice SMITH"` and `"chen  wei "` in `applicant`.

**Analogy.** Tidying a pile of handwritten name cards so they can be sorted alphabetically: same capitals, no stray marks.

**Explanation.** The text functions mirror pandas' `.str` methods:

| pandas `.str` | PySpark `F` | Does |
|---|---|---|
| `.str.strip()` | `F.trim(c)` | Remove spaces at both ends |
| `.str.upper()` / `.lower()` / `.title()` | `F.upper(c)` / `F.lower(c)` / `F.initcap(c)` | Change case |
| `.str.replace(pattern, new, regex=True)` | `F.regexp_replace(c, pattern, new)` | Replace a pattern |
| `.str.extract(pattern)` | `F.regexp_extract(c, pattern, group)` | Pull part of the text out |
| `.str.split(sep)` | `F.split(c, sep)` | Split into an array |
| `.str.len()` / `.str.contains(...)` | `F.length(c)` / `F.col(c).contains(...)` | Length / contains |

**What we want to do.**

- `product`: (1) remove the spaces at the start and end, (2) write every value with a capital first letter, so `"card"`, `"Card"` and `" Mortgage "` become `"Card"`, `"Card"` and `"Mortgage"`.
- `applicant`: (1) remove the spaces at the start and end, (2) replace any run of spaces **inside** the name with a single space, (3) capitalise each word: `"  alice SMITH"` becomes `"Alice Smith"`, `"chen  wei "` becomes `"Chen Wei"`.

First, the `applicant` steps one at a time, each in its own column so you can see what it does.

In [ ]:
# Step 1: remove the spaces at the start and the end:   "  alice SMITH" -> "alice SMITH"
# Step 2: replace every run of spaces inside the text by one space:   "chen  wei" -> "chen wei"
#         "\\s+" is a regular expression meaning "one or more spaces"
# Step 3: capital first letter for every word, the rest lower case:   "alice SMITH" -> "Alice Smith"

In practice the steps are written as **one expression**, the functions nested inside each other. Read it **from the inside out**: `F.trim` runs first, its result goes into `F.regexp_replace`, and that result into `F.initcap`.

In [ ]:
# product: trim, then capitalise        (2 steps, inside out)
# applicant: trim, then squeeze the spaces, then capitalise        (3 steps, inside out)
# Check the effect: the distinct product values before and after

Seven spellings became five values: the four real products plus `Crypto`, an unknown product: exactly the kind of value a business rule catches (Section 9). Note that `initcap` turns `O'Neil` into `O'neil`: automatic case rules are a good default, not perfect, and names are the classic exception.

In [ ]:
# Extracting part of the text
# the first word: ^ means "start of the text", (\w+) captures one or more letters; 1 = that captured group
# the surname's first letter: split on spaces, take the last word (-1), then its first character

<a id="dups"></a>
# Section 8: Duplicates

**Definition.** An **exact duplicate** repeats every column of another row. A **key duplicate** repeats the business key (here `app_id`) with different details, which is usually a correction or a data entry problem.

**Example.** A002 appears twice, identically. A003 appears twice with different amounts and decisions.

**Analogy.** Receiving the same invoice twice (exact duplicate: throw one away) versus two different invoices with the same number (key duplicate: you need a rule to decide which is right).

**Explanation.** `dropDuplicates()` (or `distinct()`) removes exact duplicates, like pandas' `drop_duplicates()`. `dropDuplicates(["app_id"])` compares only the listed columns, like `drop_duplicates(subset=["app_id"])`, and keeps **one row per key, but which one is not guaranteed**. So look at the key duplicates before removing them: if the rows differ, someone has to decide which one is right.

**What we want to do.**

1. Remove the exact duplicates, and count how many rows that removed.
2. Find the ids that still appear more than once.
3. Look at those rows.
4. Keep one row per id, once we know that is acceptable.

In [ ]:
# Step 1: exact duplicates. dropDuplicates() with no arguments compares every column.
# Step 2: key duplicates. Count the rows per app_id and keep the ids that appear more than once.

In [ ]:
# Step 3: look at the rows behind those ids (an inner join keeps only the duplicated ids)

In [ ]:
# Step 4: keep one row per app_id. Only the listed column is compared.
# Which of the two A003 rows survives is NOT guaranteed: in a real job, agree the rule with the
# business first (for example "the most recent application"), or send both rows for review.

<a id="rules"></a>
# Section 9: Rule checks: a quality report for Superstore

**Definition.** A **data rule** is a condition every valid record must satisfy, taken from the business: sales are positive, a quantity is at least 1, a discount stays within what the business allows. **Quarantining** means setting aside the records that break a rule, with the reason, instead of deleting them or letting them through.

**Example.** Before the Superstore orders feed a sales dashboard, check them and produce a short quality report.

**Analogy.** Quality control on a production line: faulty items go into a labelled box for inspection, instead of being thrown away or shipped to the customer.

**Explanation.** Three steps, all with tools from the sections above:

1. **Inspect:** count the nulls and the duplicate rows (Sections 3 and 8).
2. **Check the rules:** write each rule as a condition and combine them with `F.when` into one column, `dq_issue`, that names the first rule a row breaks (and stays null when the row is clean).
3. **Split:** two filters give the valid rows, which carry on, and the quarantined rows, which go for review. Counting `dq_issue` is a one-line quality report you can track over time.

**Step 1: read and inspect.**

In [ ]:
# Default: the pandas bridge. pandas downloads the file from the course S3 bucket over HTTPS and
# Spark converts the result. Works on every compute; fine for training-size files, never for
# real big data (the whole file passes through the driver's memory).
superstore = pandas_bridge(spark, "superstore", encoding="latin-1")

# Alternative 1: the same two steps in plain code. pandas reads the file, Spark converts it.
# (The bridge also turns pandas' missing values, NaN, into proper Spark nulls; this version does not.)
# pdf = pd.read_csv(session_datasets_http["superstore"], encoding="latin-1")
# superstore = spark.createDataFrame(pdf)
# Alternative 2: Spark reads the file straight from S3 (the big-data way; needs S3 access on your compute)
# superstore = spark.read.csv(session_datasets["superstore"], header=True, inferSchema=True, escape='"', encoding="ISO-8859-1")

In [ ]:
# Your turn. Write your solution here:

In [ ]:
# Nulls per column (Section 3). .first() takes the single result row and .asDict() turns it
# into a Python dictionary {column: number of nulls}, so we can print only the columns that have any
# Exact duplicates (Section 8): rows before minus rows after dropDuplicates

**Step 2: the rules.** Sales must be positive, the quantity at least 1, and, a business rule from the sales team, **a discount above 50% needs a manager's approval**, so those lines are set aside for review.

In [ ]:
# The quality report: how many rows break each rule (the null group = the clean rows)

**Step 3: split into valid and quarantined rows.**

In [ ]:
# Where do the quarantined lines come from? Count them by region and discount level

**Result.** No missing values, no duplicate rows, and every sale and quantity is valid. The discount rule sets aside **856 order lines (about 9%)**, most of them in the **Central** region, where discounts of 60% to 80% are common. They are not deleted: they go to the sales team with the reason attached, and the dashboard uses the valid rows until each one is confirmed. A report like this, run every time the data arrives, is the evidence that the data can be used.

<a id="exercises"></a>
# Exercises

Use the Superstore orders for 2019 (`orders_2019`, from the walkthrough), loaded below.

In [ ]:
orders_2019 = pandas_bridge(spark, "orders_2019", index_col=0)
# Alternative (plain pandas, then Spark): orders_2019 = spark.createDataFrame(pd.read_csv(session_datasets_http["orders_2019"], index_col=0))
# Alternative (Spark reads S3):
# orders_2019 = spark.read.csv(session_datasets["orders_2019"], header=True, inferSchema=True, escape='"').drop("_c0")

In [ ]:
# Your turn. Write your solution here:

### Exercise 1: Inspect

Count the nulls in every column of `orders_2019` and the exact duplicate rows. Which column has missing values, and for which city and state?

> *Hint:* `show(vertical=True)` prints one column per line, easier to read for wide tables.

In [ ]:
# Your turn

### Exercise 2: Profit as a number

`profit` is text such as `"-$1.02"` or `"$1,234.56"`. Convert it to a number in a column `profit_num` (strip `$` and `,`, then `try_cast`). How many values failed to convert, and how many order lines lost money?

In [ ]:
# Your turn

### Exercise 3: A consistency rule from text

`product_id` starts with a family code (`FUR-...`, `OFF-...`, `TEC-...`). Extract it into `family`, map it to the category names (`FUR` to Furniture, `OFF` to Office Supplies, `TEC` to Technology) and count the rows where the mapped family does not match `category`.

> *Hint:* `F.split(col, "-")` returns an array; `F.element_at(array, 1)` takes its first element. `replace` maps values with a dictionary.

In [ ]:
# Your turn

### Exercise 4: Key duplicates

Find the `order_id` + `product_id` combinations that appear more than once and show those rows. Then build `orders_2019_dedup` with one row per combination using `dropDuplicates`, and print the row counts before and after.

In [ ]:
# Your turn

<a id="takeaways"></a>
## Key Takeaways

| Step | pandas (Session 3) | PySpark |
|---|---|---|
| Nulls per column | `df.isnull().sum()` | `df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns])` |
| NaN | (same as null in pandas) | `F.isnan(c)`: `isNull()` does not catch NaN |
| Summary | `df.describe()` | `df.summary()` / `df.describe()` |
| Drop missing | `dropna(subset=...)` | `df.na.drop(subset=[...])` |
| Fill missing | `fillna({...})` | `df.na.fill({...})`; median via `F.percentile_approx(c, 0.5)` |
| Convert types | `pd.to_numeric(errors="coerce")` | `F.expr("try_cast(c AS DOUBLE)")`, `try_to_timestamp` |
| Transform values | `.apply`, `.map`, `.replace`, `pd.cut` | `F.when(...).otherwise(...)`, `df.replace({...}, subset=[...])` |
| Clean text | `.str.strip/upper/replace/extract` | `F.trim`, `F.upper`, `F.initcap`, `F.regexp_replace`, `F.regexp_extract` |
| Exact duplicates | `drop_duplicates()` | `dropDuplicates()` |
| Key duplicates | `drop_duplicates(subset=[key])` | `groupBy(key).count()` to find them, then `dropDuplicates([key])` (which row is kept is not guaranteed) |
| Rules | boolean masks | `F.when(rule, "reason")...`, then split valid / quarantined |

<a id="conclusion"></a>
## Conclusion

You now have the Session 3 cleaning routine in Spark: inspect, handle missing values, convert types safely, transform and clean values with built-in functions, remove duplicates deliberately, and quarantine records that break business rules. The practical (02.03) applies it to the Titanic passengers and prepares them for the MLlib models; notebook 02.04 then looks at what Spark does **under the hood** and how to make it faster.

<a id="reading"></a>
## Further Reading & Resources

- Apache Spark: [DataFrameNaFunctions (`na.drop`, `na.fill`, `na.replace`)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameNaFunctions.html)
- Apache Spark: [DataFrame.dropDuplicates](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html)
- Apache Spark: [String functions (`trim`, `regexp_replace`, `regexp_extract`, ...)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#string-functions)
- Apache Spark: [percentile_approx](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.percentile_approx.html)